# DeepSets vs Graph Neural Network Comparison

## Research-Backed Methodology

This notebook implements a comprehensive, research-backed comparison between DeepSets (feedforward MLP baseline) and GraphSAGE (GNN) models trained on surface code error correction. The comparison follows established ML benchmarking practices from Nature Scientific Reports, JMLR, ICLR/NeurIPS, and IEEE publications.

**Key Principles:**
- Fair comparison with parameter budget matching
- Statistical significance testing (not just mean accuracy)
- Identical evaluation protocols
- Multiple performance indices (accuracy, speed, memory, complexity)
- Proper baseline comparison (DeepSets as structure-agnostic baseline)

**Distances Analyzed:** d=3, d=5, d=7, d=9, d=11, d=13

## 1. Imports and Setup

In [ ]:
import sys
import json
import time
import gc
import os
from pathlib import Path
from datetime import datetime
from typing import Dict, List, Tuple, Optional
import warnings
warnings.filterwarnings('ignore')

import sklearn
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch_geometric.data import Batch, Data
from torch_geometric.loader import DataLoader
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
from scipy import stats
from scipy.stats import wilcoxon, chi2_contingency
import math
import stim

# Set style for plots
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

# Detect if running in Google Colab
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_PATH = Path('/content/drive/MyDrive/Research/QEC/quantum-error-correction/quantum-error-correction/code')
else:
    BASE_PATH = Path('../..')  # code/results/deepsets_vs_gnn_comparison -> code/

sys.path.insert(0, str(BASE_PATH))

# Import model classes
from benchmark_models import DeepSets, SurfaceCodeSampler
from models import GraphSAGE, SparseGraph

# Set up paths
RESULTS_DIR = BASE_PATH / "results" / "deepsets_vs_gnn_comparison"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_RESULTS_DIR = RESULTS_DIR / "results"
OUTPUT_RESULTS_DIR.mkdir(parents=True, exist_ok=True)
PLOTS_DIR = RESULTS_DIR / "plots"
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

# Device setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

print(f"\nPaths:")
print(f"  BASE_PATH: {BASE_PATH}")
print(f"  RESULTS_DIR: {RESULTS_DIR}")
print(f"  OUTPUT_RESULTS_DIR: {OUTPUT_RESULTS_DIR}")
print(f"  PLOTS_DIR: {PLOTS_DIR}")

## 2. Configuration

In [ ]:
# ============================================================
# DATA MODE CONFIGURATION
# ============================================================
# Set to True to use pre-saved CSV data from /plots/final/
# Set to False to regenerate all data from scratch (runs full benchmark)
USE_SAVED_DATA = False

# Final plots directory
FINAL_PLOTS_DIR = PLOTS_DIR / "final"
FINAL_PLOTS_DIR.mkdir(parents=True, exist_ok=True)

# ============================================================
# EXPERIMENT CONFIGURATION
# ============================================================
# Distances to compare
DISTANCES = [3, 5, 7, 9, 11, 13]

# Test dataset configuration
TEST_SAMPLES_PER_DISTANCE = 10000  # Per distance for evaluation
BENCHMARK_SAMPLES = 10000  # For inference speed benchmarking

# Inference benchmarking configuration
BATCH_SIZES = [1, 8, 16, 32, 64, 128, 256]
NUM_WARMUP_RUNS = 10  # Warmup batches before timing
NUM_BENCHMARK_RUNS = 5  # Number of independent runs for statistical robustness
NUM_ACCURACY_RUNS = 4  # Number of independent accuracy evaluations

# Statistical test significance level
ALPHA = 0.05

# Random seed for reproducibility
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

print(f"Configuration:")
print(f"  USE_SAVED_DATA: {USE_SAVED_DATA}")
print(f"  FINAL_PLOTS_DIR: {FINAL_PLOTS_DIR}")
print(f"  Distances: {DISTANCES}")
print(f"  Test samples per distance: {TEST_SAMPLES_PER_DISTANCE:,}")
print(f"  Benchmark samples: {BENCHMARK_SAMPLES:,}")
print(f"  Batch sizes for benchmarking: {BATCH_SIZES}")
print(f"  Number of benchmark runs: {NUM_BENCHMARK_RUNS}")
print(f"  Number of accuracy runs: {NUM_ACCURACY_RUNS}")
print(f"  Significance level (alpha): {ALPHA}")
print(f"  Random seed: {SEED}")

## 3. Helper Functions

In [ ]:
def count_parameters(model: nn.Module) -> int:
    """Count trainable parameters in a model."""
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


def get_model_size_mb(model_path: Path) -> float:
    """Get model file size in MB."""
    if model_path.exists():
        return model_path.stat().st_size / (1024 * 1024)
    return 0.0


# Model names for extrapolation experiments (16 total)
MODEL_NAMES = [
    "a1_d3_00", "a2_d3_10", "a3_d3_20", "a4_d3_40", "a5_d3_50",
    "b1_d5heavy", "b2_d5more", "b3_balanced", "b4_d7more", "b5_d7heavy",
    "c1_only_d3", "c2_only_d5", "c3_only_d7", "c4_no_d7",
    "equal_333333"
]


def load_deepsets_model(model_name: str, base_path: Path) -> Tuple[DeepSets, Dict]:
    """Load DeepSets model for a given model name."""
    model_path = base_path / "deepsets" / "extrapolation" / "models" / "revised_training" / f"{model_name}.pt"

    if not model_path.exists():
        raise FileNotFoundError(f"DeepSets model not found: {model_name} at {model_path}")

    checkpoint = torch.load(model_path, map_location=device, weights_only=False)
    config = checkpoint.get('config', {})

    model = DeepSets(
        nickname=model_name,
        phi_hidden=config.get('phi_hidden', [64, 128, 256]),
        rho_hidden=config.get('rho_hidden', [256, 128, 64]),
        pool=config.get('pool', 'mean'),
        dropout=config.get('dropout', 0.1),
        device=device,
        base_path=base_path
    )

    if 'state_dict' in checkpoint:
        model.model.load_state_dict(checkpoint['state_dict'])

    model.model.eval()
    model._config = config

    info = {
        'name': model_name,
        'num_parameters': count_parameters(model.model),
        'model_size_mb': get_model_size_mb(model_path),
        'model_path': str(model_path),
        'config': config
    }

    return model, info


def load_graphsage_model(model_name: str, base_path: Path) -> Tuple[GraphSAGE, Dict]:
    """Load GraphSAGE model for a given model name."""
    model_path = base_path / "gSAGE" / "extrapolation" / "models" / "revised_training" / f"{model_name}.pt"

    if not model_path.exists():
        raise FileNotFoundError(f"GraphSAGE model not found: {model_name} at {model_path}")

    checkpoint = torch.load(model_path, map_location=device, weights_only=False)
    config = checkpoint.get('config', {
        'in_channels': 5, 'hidden_dim': 128, 'num_layers': 5, 'dropout': 0.0, 'aggr': 'max'
    })

    model = GraphSAGE(
        nickname=model_name,
        in_channels=config.get('in_channels', 5),
        hidden_dim=config.get('hidden_dim', 128),
        num_layers=config.get('num_layers', 5),
        dropout=config.get('dropout', 0.0),
        aggr=config.get('aggr', 'max'),
        device=device,
        base_path=base_path
    )

    if 'state_dict' in checkpoint:
        model.model.load_state_dict(checkpoint['state_dict'])

    model.model.eval()

    info = {
        'name': model_name,
        'num_parameters': count_parameters(model.model),
        'model_size_mb': get_model_size_mb(model_path),
        'model_path': str(model_path),
        'config': config
    }

    return model, info


print("Helper functions defined.")

In [ ]:
# Load all models
deepsets_models = {}
graphsage_models = {}
deepsets_info = {}
graphsage_info = {}

print("Loading models...")
for model_name in MODEL_NAMES:
    # Load DeepSets
    try:
        model, info = load_deepsets_model(model_name, BASE_PATH)
        deepsets_models[model_name] = model
        deepsets_info[model_name] = info
        print(f"  ✓ DeepSets {model_name}: {info['num_parameters']:,} params, {info['model_size_mb']:.2f} MB")
    except Exception as e:
        print(f"  ✗ DeepSets {model_name}: Failed to load - {e}")

    # Load GraphSAGE
    try:
        model, info = load_graphsage_model(model_name, BASE_PATH)
        graphsage_models[model_name] = model
        graphsage_info[model_name] = info
        print(f"  ✓ GraphSAGE {model_name}: {info['num_parameters']:,} params, {info['model_size_mb']:.2f} MB")
    except Exception as e:
        print(f"  ✗ GraphSAGE {model_name}: Failed to load - {e}")

print(f"\nLoaded {len(deepsets_models)} DeepSets models and {len(graphsage_models)} GraphSAGE models")

## 5. Load Training Results

In [ ]:
# Load DeepSets training results from revised_training folder
deepsets_training_results = {}
deepsets_results_path = BASE_PATH / "deepsets" / "training" / "results" / "revised_training" / "results.json"

if deepsets_results_path.exists():
    with open(deepsets_results_path, 'r') as f:
        deepsets_results = json.load(f)

    for result in deepsets_results:
        # Include both 'completed' and 'completed blowup' status
        if result.get('status', '').startswith('completed'):
            d = result['distance']
            deepsets_training_results[d] = result
    print(f"Loaded DeepSets training results for {len(deepsets_training_results)} distances")
else:
    print(f"Warning: DeepSets results file not found at {deepsets_results_path}")

# Load GraphSAGE training results from revised_training folder
graphsage_training_results = {}
for d in DISTANCES:
    results_path = BASE_PATH / "gSAGE" / "distances" / "results" / "revised_training" / f"d{d}_training.json"
    if results_path.exists():
        with open(results_path, 'r') as f:
            graphsage_training_results[d] = json.load(f)
    else:
        print(f"Warning: GraphSAGE results file not found for d={d} at {results_path}")

print(f"Loaded GraphSAGE training results for {len(graphsage_training_results)} distances")

## 6. Parameter Budget Analysis

In [ ]:
# Create parameter comparison table (for each model)
param_comparison = []
for model_name in MODEL_NAMES:
    if model_name in deepsets_info and model_name in graphsage_info:
        param_comparison.append({
            'model_name': model_name,
            'deepsets_params': deepsets_info[model_name]['num_parameters'],
            'graphsage_params': graphsage_info[model_name]['num_parameters'],
            'deepsets_size_mb': deepsets_info[model_name]['model_size_mb'],
            'graphsage_size_mb': graphsage_info[model_name]['model_size_mb'],
            'param_ratio': graphsage_info[model_name]['num_parameters'] / deepsets_info[model_name]['num_parameters'] if deepsets_info[model_name]['num_parameters'] > 0 else 0
        })

param_df = pd.DataFrame(param_comparison)
print("Parameter Budget Comparison:")
print(param_df.to_string(index=False))

if len(param_comparison) > 0:
    avg_ratio = param_df['param_ratio'].mean()
    print(f"\nAverage parameter ratio (GraphSAGE/DeepSets): {avg_ratio:.2f}x")
    print(f"DeepSets avg params: {param_df['deepsets_params'].mean():,.0f}")
    print(f"GraphSAGE avg params: {param_df['graphsage_params'].mean():,.0f}")

    if avg_ratio > 1.5 or avg_ratio < 0.67:
        print(f"\n⚠️  WARNING: Significant parameter budget difference ({avg_ratio:.2f}x)")
        print("   This should be documented in the comparison.")
    else:
        print(f"\n✓ Parameter budgets are reasonably matched ({avg_ratio:.2f}x)")

## 7. Generate Shared Test Sets (Fair Comparison Protocol)

In [ ]:
# Generate shared test datasets for fair comparison
# Same random seed ensures identical data for both models
shared_test_data = {}
graph_builder = SparseGraph(k_neighbors=6, device=torch.device('cpu'))
sampler = SurfaceCodeSampler(p=0.005, device=torch.device('cpu'))

print("Generating shared test datasets...")
for d in DISTANCES:
    # Generate detections with fixed seed for reproducibility
    torch.manual_seed(SEED + d)
    np.random.seed(SEED + d)

    detections, labels = sampler.sample(
        d=d,
        num_samples=TEST_SAMPLES_PER_DISTANCE,
        p_values=[0.001, 0.003, 0.005, 0.007],
        p_weights=[0.25, 0.25, 0.25, 0.25]
    )

    graphs = graph_builder.batch_to_pyg(detections, labels)

    shared_test_data[d] = {
        'detections': detections.cpu(),
        'labels': labels.cpu(),
        'graphs': graphs,
        'num_samples': len(labels)
    }

    print(f"  ✓ d={d}: {len(labels):,} samples")

print(f"\nGenerated shared test data for {len(shared_test_data)} distances")

## 8. Inference Speed Benchmarking

In [ ]:
def benchmark_inference_speed(
    model,
    data,
    batch_sizes: List[int],
    num_warmup: int = 10,
    num_runs: int = 5,
    is_graph_model: bool = False,
    distance: int = None
) -> Dict:
    """Benchmark inference speed with multiple runs for statistical robustness."""
    model.model.eval()
    model.model.to(device)

    results = {}

    for batch_size in batch_sizes:
        if is_graph_model:
            loader = DataLoader(data, batch_size=batch_size, shuffle=False)
            num_samples = len(data)
        else:
            num_samples = len(data)

        # Warmup
        with torch.no_grad():
            warmup_count = 0
            if is_graph_model:
                for batch in loader:
                    if warmup_count >= num_warmup:
                        break
                    batch = batch.to(device)
                    _ = model.model(batch)
                    warmup_count += 1
            else:
                for i in range(0, min(num_warmup * batch_size, num_samples), batch_size):
                    batch = data[i:i+batch_size].to(device)
                    _ = model.predict(batch, distance)
                    warmup_count += 1

        if device.type == 'cuda':
            torch.cuda.synchronize()

        run_times = []
        for run in range(num_runs):
            if device.type == 'cuda':
                torch.cuda.synchronize()

            start_time = time.time()

            with torch.no_grad():
                if is_graph_model:
                    for batch in loader:
                        batch = batch.to(device)
                        _ = model.model(batch)
                else:
                    for i in range(0, num_samples, batch_size):
                        batch = data[i:i+batch_size].to(device)
                        _ = model.predict(batch, distance)

            if device.type == 'cuda':
                torch.cuda.synchronize()

            elapsed = time.time() - start_time
            run_times.append(elapsed)

        mean_time = np.mean(run_times)
        std_time = np.std(run_times)
        throughput = num_samples / mean_time
        latency_per_sample = (mean_time / num_samples) * 1e6

        results[batch_size] = {
            'mean_time_seconds': mean_time,
            'std_time_seconds': std_time,
            'throughput_samples_per_sec': throughput,
            'latency_per_sample_us': latency_per_sample,
            'runs': run_times
        }

    return results


# Benchmark all models across all distances
print("Benchmarking inference speed...")
inference_benchmarks = {}

for model_name in MODEL_NAMES:
    if model_name not in deepsets_models or model_name not in graphsage_models:
        continue

    inference_benchmarks[model_name] = {}

    for d in DISTANCES:
        if d not in shared_test_data:
            continue

        if model_name not in inference_benchmarks:
            inference_benchmarks[model_name] = {}
        if d not in inference_benchmarks[model_name]:
            inference_benchmarks[model_name][d] = {}

        # Benchmark DeepSets
        print(f"  Benchmarking DeepSets {model_name} d={d}...")
        detections = shared_test_data[d]['detections']
        results = benchmark_inference_speed(
            deepsets_models[model_name],
            detections,
            BATCH_SIZES,
            NUM_WARMUP_RUNS,
            NUM_BENCHMARK_RUNS,
            is_graph_model=False,
            distance=d
        )
        inference_benchmarks[model_name][d]['deepsets'] = results

        # Benchmark GraphSAGE
        print(f"  Benchmarking GraphSAGE {model_name} d={d}...")
        graphs = shared_test_data[d]['graphs']
        results = benchmark_inference_speed(
            graphsage_models[model_name],
            graphs,
            BATCH_SIZES,
            NUM_WARMUP_RUNS,
            NUM_BENCHMARK_RUNS,
            is_graph_model=True
        )
        inference_benchmarks[model_name][d]['graphsage'] = results

print("\n✓ Inference benchmarking complete")

In [ ]:
def evaluate_accuracy_metrics(
    model,
    data,
    labels,
    num_runs: int = 4,
    threshold: float = 0.5,
    is_graph_model: bool = False,
    batch_size: int = 256,
    distance: int = None
) -> Dict:
    """Evaluate accuracy metrics with multiple runs for statistical robustness."""
    model.model.eval()
    model.model.to(device)

    all_accuracies = []
    all_precisions = []
    all_recalls = []
    all_f1s = []
    all_predictions = []

    for run in range(num_runs):
        with torch.no_grad():
            predictions = []

            if is_graph_model:
                loader = DataLoader(data, batch_size=batch_size, shuffle=False)
                for batch in loader:
                    batch = batch.to(device)
                    pred = model.model(batch)
                    predictions.append(pred.cpu())
            else:
                for i in range(0, len(data), batch_size):
                    batch = data[i:i+batch_size].to(device)
                    pred = model.predict(batch, distance)
                    predictions.append(pred.cpu())

            predictions = torch.cat(predictions, dim=0).squeeze()
            binary_preds = (predictions >= threshold).float()
            labels_tensor = labels.float()

            correct = (binary_preds == labels_tensor).sum().item()
            accuracy = correct / len(labels_tensor)

            tp = ((binary_preds == 1) & (labels_tensor == 1)).sum().item()
            fp = ((binary_preds == 1) & (labels_tensor == 0)).sum().item()
            fn = ((binary_preds == 0) & (labels_tensor == 1)).sum().item()

            precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
            recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
            f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0.0

            all_accuracies.append(accuracy)
            all_precisions.append(precision)
            all_recalls.append(recall)
            all_f1s.append(f1)

            if run == 0:
                all_predictions = binary_preds.numpy()

    return {
        'accuracy_mean': np.mean(all_accuracies),
        'accuracy_std': np.std(all_accuracies),
        'accuracy_ci_95': 1.96 * np.std(all_accuracies) / np.sqrt(num_runs),
        'precision_mean': np.mean(all_precisions),
        'precision_std': np.std(all_precisions),
        'recall_mean': np.mean(all_recalls),
        'recall_std': np.std(all_recalls),
        'f1_mean': np.mean(all_f1s),
        'f1_std': np.std(all_f1s),
        'f1_ci_95': 1.96 * np.std(all_f1s) / np.sqrt(num_runs),
        'all_accuracies': all_accuracies,
        'all_f1s': all_f1s,
        'predictions': all_predictions
    }


# Evaluate all models on all distances
print("Evaluating accuracy metrics...")
accuracy_results = {}

for model_name in tqdm(MODEL_NAMES, desc="Models"):
    if model_name not in deepsets_models or model_name not in graphsage_models:
        continue

    accuracy_results[model_name] = {}

    for d in DISTANCES:
        if d not in shared_test_data:
            continue

        accuracy_results[model_name][d] = {}
        labels = shared_test_data[d]['labels']

        # Evaluate DeepSets
        detections = shared_test_data[d]['detections']
        results = evaluate_accuracy_metrics(
            deepsets_models[model_name],
            detections,
            labels,
            NUM_ACCURACY_RUNS,
            is_graph_model=False,
            distance=d
        )
        accuracy_results[model_name][d]['deepsets'] = results

        # Evaluate GraphSAGE
        graphs = shared_test_data[d]['graphs']
        results = evaluate_accuracy_metrics(
            graphsage_models[model_name],
            graphs,
            labels,
            NUM_ACCURACY_RUNS,
            is_graph_model=True
        )
        accuracy_results[model_name][d]['graphsage'] = results

print("\n✓ Accuracy evaluation complete")

In [ ]:
def wilcoxon_test(deepsets_accs: List[float], graphsage_accs: List[float]) -> Dict:
    """Perform Wilcoxon signed-rank test."""
    if len(deepsets_accs) < 2 or len(graphsage_accs) < 2:
        return {'test': 'Wilcoxon', 'p_value': 1.0, 'significant': False, 'note': 'Insufficient samples'}

    statistic, p_value = wilcoxon(deepsets_accs, graphsage_accs, alternative='two-sided')
    n = len(deepsets_accs)
    z = stats.norm.ppf(p_value / 2) if p_value > 0 else 0
    r = abs(z) / np.sqrt(n)

    return {
        'test': 'Wilcoxon signed-rank',
        'statistic': float(statistic),
        'p_value': float(p_value),
        'significant': p_value < ALPHA,
        'effect_size_r': float(r),
        'interpretation': 'large' if r > 0.5 else ('medium' if r > 0.3 else 'small')
    }


def cohens_d(group1: List[float], group2: List[float]) -> float:
    """Calculate Cohen's d effect size."""
    n1, n2 = len(group1), len(group2)
    mean1, mean2 = np.mean(group1), np.mean(group2)
    var1, var2 = np.var(group1, ddof=1), np.var(group2, ddof=1)

    pooled_std = np.sqrt(((n1 - 1) * var1 + (n2 - 1) * var2) / (n1 + n2 - 2))

    if pooled_std == 0:
        return 0.0

    return float((mean1 - mean2) / pooled_std)


# Perform statistical tests for each model
print("Performing statistical tests...")
statistical_tests = {}

for model_name in MODEL_NAMES:
    if model_name not in accuracy_results:
        continue

    deepsets_all_accs = []
    graphsage_all_accs = []

    for d in DISTANCES:
        if d not in accuracy_results[model_name]:
            continue

        if 'deepsets' in accuracy_results[model_name][d]:
            deepsets_all_accs.extend(accuracy_results[model_name][d]['deepsets']['all_accuracies'])
        if 'graphsage' in accuracy_results[model_name][d]:
            graphsage_all_accs.extend(accuracy_results[model_name][d]['graphsage']['all_accuracies'])

    if len(deepsets_all_accs) >= 2 and len(graphsage_all_accs) >= 2:
        # Ensure equal length for paired test
        min_len = min(len(deepsets_all_accs), len(graphsage_all_accs))
        deepsets_paired = deepsets_all_accs[:min_len]
        graphsage_paired = graphsage_all_accs[:min_len]

        wilcoxon_result = wilcoxon_test(deepsets_paired, graphsage_paired)
        cohens_d_val = cohens_d(deepsets_paired, graphsage_paired)

        statistical_tests[model_name] = {
            'wilcoxon': wilcoxon_result,
            'cohens_d': cohens_d_val,
            'effect_size_interpretation': 'large' if abs(cohens_d_val) > 0.8 else ('medium' if abs(cohens_d_val) > 0.5 else 'small'),
            'deepsets_mean': np.mean(deepsets_paired),
            'graphsage_mean': np.mean(graphsage_paired),
            'diff': np.mean(graphsage_paired) - np.mean(deepsets_paired)
        }
        print(f"  {model_name}: DS={np.mean(deepsets_paired):.4f}, GS={np.mean(graphsage_paired):.4f}, p={wilcoxon_result['p_value']:.4f}")

print("\n✓ Statistical testing complete")

## 11. Training Convergence Analysis

In [ ]:
# Extract training convergence data
training_convergence = {}

for d in DISTANCES:
    training_convergence[d] = {}

    # DeepSets training data
    if d in deepsets_training_results:
        result = deepsets_training_results[d]
        if 'epoch_metrics' in result:
            epoch_metrics = result['epoch_metrics']
            training_convergence[d]['deepsets'] = {
                'epochs': epoch_metrics.get('epoch', []),
                'train_loss': epoch_metrics.get('train_loss', []),
                'val_accuracy': epoch_metrics.get('val_accuracy', []),
                'train_accuracy': epoch_metrics.get('train_accuracy', []),
                'training_time_seconds': result.get('training_time_seconds', 0),
                'final_test_accuracy': result.get('test_accuracy', 0)
            }
        else:
            # Fallback to basic info
            training_convergence[d]['deepsets'] = {
                'final_test_accuracy': result.get('test_accuracy', 0),
                'training_time_seconds': result.get('training_time_seconds', 0),
                'epochs': result.get('epochs', 0)
            }

    # GraphSAGE training data
    if d in graphsage_training_results:
        result = graphsage_training_results[d]
        if 'epoch_metrics' in result:
            epoch_metrics = result['epoch_metrics']
            training_convergence[d]['graphsage'] = {
                'epochs': epoch_metrics.get('epochs', []),
                'train_loss': epoch_metrics.get('train_loss', []),
                'val_accuracy': epoch_metrics.get('val_accuracy', []),
                'train_accuracy': epoch_metrics.get('train_accuracy', []),
                'training_time_seconds': result.get('training_config', {}).get('epochs', 0) * (
                    epoch_metrics.get('epoch_time_seconds', [0])[0] if epoch_metrics.get('epoch_time_seconds') else 0
                ),
                'final_test_accuracy': result.get('final_metrics', {}).get('test_accuracy', 0) if 'final_metrics' in result else 0
            }
        else:
            training_convergence[d]['graphsage'] = {
                'final_test_accuracy': 0,
                'training_time_seconds': 0,
                'epochs': result.get('training_config', {}).get('epochs', 0)
            }

print("Training convergence data extracted")

## 12. Visualization Plots

In [ ]:
def plot_accuracy_vs_distance(accuracy_results, statistical_tests, output_path):
    """Graph 1: Accuracy vs Code Distance with error bars."""
    distances = sorted([d for d in accuracy_results.keys() if 'deepsets' in accuracy_results[d] and 'graphsage' in accuracy_results[d]])

    deepsets_accs = []
    deepsets_cis = []
    graphsage_accs = []
    graphsage_cis = []
    significance = []

    for d in distances:
        deepsets_accs.append(accuracy_results[d]['deepsets']['accuracy_mean'])
        deepsets_cis.append(accuracy_results[d]['deepsets']['accuracy_ci_95'])
        graphsage_accs.append(accuracy_results[d]['graphsage']['accuracy_mean'])
        graphsage_cis.append(accuracy_results[d]['graphsage']['accuracy_ci_95'])

        # Check significance
        if d in statistical_tests:
            sig = statistical_tests[d]['wilcoxon']['significant']
            significance.append('*' if sig else '')
        else:
            significance.append('')

    fig, ax = plt.subplots(figsize=(10, 6))

    x = np.arange(len(distances))
    width = 0.35

    # Plot with error bars
    bars1 = ax.errorbar(x - width/2, deepsets_accs, yerr=deepsets_cis,
                       fmt='o-', label='DeepSets', color='steelblue',
                       capsize=5, capthick=2, linewidth=2, markersize=8)
    bars2 = ax.errorbar(x + width/2, graphsage_accs, yerr=graphsage_cis,
                       fmt='s-', label='GraphSAGE', color='coral',
                       capsize=5, capthick=2, linewidth=2, markersize=8)

    # Add significance markers
    for i, (d, sig) in enumerate(zip(distances, significance)):
        if sig:
            max_acc = max(deepsets_accs[i] + deepsets_cis[i], graphsage_accs[i] + graphsage_cis[i])
            ax.text(i, max_acc + 0.02, sig, ha='center', fontsize=14, fontweight='bold')

    ax.set_xlabel('Code Distance (d)', fontsize=12, fontweight='bold')
    ax.set_ylabel('Accuracy', fontsize=12, fontweight='bold')
    ax.set_title('Accuracy vs Code Distance (95% CI)', fontsize=14, fontweight='bold')
    ax.set_xticks(x)
    ax.set_xticklabels([f'd={d}' for d in distances])
    ax.legend(fontsize=11)
    ax.grid(True, alpha=0.3)
    ax.set_ylim([0, 1.1])

    plt.tight_layout()
    plt.savefig(output_path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"Saved: {output_path}")


def plot_pareto_frontier(accuracy_results, inference_benchmarks, output_path):
    """Graph 2: Pareto Frontier (Accuracy-Speed Tradeoff)."""
    fig, ax = plt.subplots(figsize=(10, 7))

    # Use batch size 64 for latency (representative)
    batch_size = 64

    for d in sorted(accuracy_results.keys()):
        if 'deepsets' not in accuracy_results[d] or 'graphsage' not in accuracy_results[d]:
            continue

        if d not in inference_benchmarks:
            continue

        # DeepSets
        if 'deepsets' in inference_benchmarks[d] and batch_size in inference_benchmarks[d]['deepsets']:
            deepsets_latency = inference_benchmarks[d]['deepsets'][batch_size]['latency_per_sample_us']
            deepsets_acc = accuracy_results[d]['deepsets']['accuracy_mean']
            ax.scatter(deepsets_latency, deepsets_acc, s=200, marker='o',
                      color='steelblue', alpha=0.7, edgecolors='black', linewidth=1.5)
            ax.annotate(f'DS d={d}', (deepsets_latency, deepsets_acc),
                       xytext=(5, 5), textcoords='offset points', fontsize=9)

        # GraphSAGE
        if 'graphsage' in inference_benchmarks[d] and batch_size in inference_benchmarks[d]['graphsage']:
            graphsage_latency = inference_benchmarks[d]['graphsage'][batch_size]['latency_per_sample_us']
            graphsage_acc = accuracy_results[d]['graphsage']['accuracy_mean']
            ax.scatter(graphsage_latency, graphsage_acc, s=200, marker='s',
                      color='coral', alpha=0.7, edgecolors='black', linewidth=1.5)
            ax.annotate(f'GS d={d}', (graphsage_latency, graphsage_acc),
                       xytext=(5, -15), textcoords='offset points', fontsize=9)

    ax.set_xlabel('Inference Latency (μs per sample)', fontsize=12, fontweight='bold')
    ax.set_ylabel('Accuracy', fontsize=12, fontweight='bold')
    ax.set_title('Pareto Frontier: Accuracy vs Inference Latency', fontsize=14, fontweight='bold')
    ax.grid(True, alpha=0.3)
    ax.set_xscale('log')

    # Add legend
    from matplotlib.patches import Patch
    legend_elements = [
        Patch(facecolor='steelblue', edgecolor='black', label='DeepSets'),
        Patch(facecolor='coral', edgecolor='black', label='GraphSAGE')
    ]
    ax.legend(handles=legend_elements, fontsize=11)

    plt.tight_layout()
    plt.savefig(output_path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"Saved: {output_path}")


def plot_convergence_curves(training_convergence, output_path):
    """Graph 3: Training Convergence Curves."""
    # Find distances with both models and epoch data
    distances_with_data = []
    for d in DISTANCES:
        if d in training_convergence:
            has_deepsets = 'deepsets' in training_convergence[d] and 'epochs' in training_convergence[d]['deepsets']
            has_graphsage = 'graphsage' in training_convergence[d] and 'epochs' in training_convergence[d]['graphsage']
            if has_deepsets or has_graphsage:
                distances_with_data.append(d)

    if len(distances_with_data) == 0:
        print("No convergence data available for plotting")
        return

    n_distances = len(distances_with_data)
    fig, axes = plt.subplots(1, min(n_distances, 3), figsize=(5*min(n_distances, 3), 5))
    if n_distances == 1:
        axes = [axes]

    for idx, d in enumerate(distances_with_data[:3]):  # Plot up to 3 distances
        ax = axes[idx] if n_distances > 1 else axes[0]

        # DeepSets
        if 'deepsets' in training_convergence[d] and 'epochs' in training_convergence[d]['deepsets']:
            epochs = training_convergence[d]['deepsets']['epochs']
            val_acc = training_convergence[d]['deepsets'].get('val_accuracy', [])
            if len(val_acc) > 0:
                ax.plot(epochs, val_acc, 'o-', label='DeepSets', color='steelblue', linewidth=2, markersize=4)

        # GraphSAGE
        if 'graphsage' in training_convergence[d] and 'epochs' in training_convergence[d]['graphsage']:
            epochs = training_convergence[d]['graphsage']['epochs']
            val_acc = training_convergence[d]['graphsage'].get('val_accuracy', [])
            if len(val_acc) > 0:
                ax.plot(epochs, val_acc, 's-', label='GraphSAGE', color='coral', linewidth=2, markersize=4)

        ax.set_xlabel('Epoch', fontsize=11)
        ax.set_ylabel('Validation Accuracy', fontsize=11)
        ax.set_title(f'Training Convergence - d={d}', fontsize=12, fontweight='bold')
        ax.legend(fontsize=10)
        ax.grid(True, alpha=0.3)
        ax.set_ylim([0, 1])

    plt.suptitle('Training Convergence Comparison', fontsize=14, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.savefig(output_path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"Saved: {output_path}")


def plot_throughput_vs_batchsize(inference_benchmarks, output_path):
    """Graph 4: Inference Throughput vs Batch Size."""
    fig, ax = plt.subplots(figsize=(10, 6))

    batch_sizes = sorted(BATCH_SIZES)

    # Aggregate across distances
    deepsets_throughputs = {bs: [] for bs in batch_sizes}
    graphsage_throughputs = {bs: [] for bs in batch_sizes}

    for d in inference_benchmarks.keys():
        if 'deepsets' in inference_benchmarks[d]:
            for bs in batch_sizes:
                if bs in inference_benchmarks[d]['deepsets']:
                    deepsets_throughputs[bs].append(inference_benchmarks[d]['deepsets'][bs]['throughput_samples_per_sec'])

        if 'graphsage' in inference_benchmarks[d]:
            for bs in batch_sizes:
                if bs in inference_benchmarks[d]['graphsage']:
                    graphsage_throughputs[bs].append(inference_benchmarks[d]['graphsage'][bs]['throughput_samples_per_sec'])

    # Calculate mean and std
    deepsets_means = [np.mean(deepsets_throughputs[bs]) if deepsets_throughputs[bs] else 0 for bs in batch_sizes]
    deepsets_stds = [np.std(deepsets_throughputs[bs]) if deepsets_throughputs[bs] else 0 for bs in batch_sizes]
    graphsage_means = [np.mean(graphsage_throughputs[bs]) if graphsage_throughputs[bs] else 0 for bs in batch_sizes]
    graphsage_stds = [np.std(graphsage_throughputs[bs]) if graphsage_throughputs[bs] else 0 for bs in batch_sizes]

    ax.errorbar(batch_sizes, deepsets_means, yerr=deepsets_stds,
               fmt='o-', label='DeepSets', color='steelblue',
               capsize=5, capthick=2, linewidth=2, markersize=8)
    ax.errorbar(batch_sizes, graphsage_means, yerr=graphsage_stds,
               fmt='s-', label='GraphSAGE', color='coral',
               capsize=5, capthick=2, linewidth=2, markersize=8)

    ax.set_xlabel('Batch Size', fontsize=12, fontweight='bold')
    ax.set_ylabel('Throughput (samples/second)', fontsize=12, fontweight='bold')
    ax.set_title('Inference Throughput vs Batch Size', fontsize=14, fontweight='bold')
    ax.set_xscale('log', base=2)
    ax.set_yscale('log')
    ax.set_xticks(batch_sizes)
    ax.set_xticklabels([str(bs) for bs in batch_sizes])
    ax.legend(fontsize=11)
    ax.grid(True, alpha=0.3, which='both')

    plt.tight_layout()
    plt.savefig(output_path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"Saved: {output_path}")


def plot_critical_difference(accuracy_results, statistical_tests, output_path):
    """Graph 5: Critical Difference Diagram."""
    # Rank models by accuracy across distances
    distances = sorted([d for d in accuracy_results.keys() if 'deepsets' in accuracy_results[d] and 'graphsage' in accuracy_results[d]])

    model_ranks = {'DeepSets': [], 'GraphSAGE': []}

    for d in distances:
        deepsets_acc = accuracy_results[d]['deepsets']['accuracy_mean']
        graphsage_acc = accuracy_results[d]['graphsage']['accuracy_mean']

        # Rank: lower rank = better (higher accuracy)
        if deepsets_acc > graphsage_acc:
            model_ranks['DeepSets'].append(1)
            model_ranks['GraphSAGE'].append(2)
        elif graphsage_acc > deepsets_acc:
            model_ranks['DeepSets'].append(2)
            model_ranks['GraphSAGE'].append(1)
        else:
            model_ranks['DeepSets'].append(1.5)
            model_ranks['GraphSAGE'].append(1.5)

    avg_ranks = {model: np.mean(ranks) for model, ranks in model_ranks.items()}

    fig, ax = plt.subplots(figsize=(8, 4))

    models = list(avg_ranks.keys())
    ranks = [avg_ranks[m] for m in models]

    # Plot ranked bars
    colors = ['steelblue' if m == 'DeepSets' else 'coral' for m in models]
    bars = ax.barh(models, ranks, color=colors, edgecolor='black', linewidth=1.5)

    # Add rank values
    for i, (model, rank) in enumerate(zip(models, ranks)):
        ax.text(rank, i, f' {rank:.2f}', va='center', fontsize=11, fontweight='bold')

    ax.set_xlabel('Average Rank', fontsize=12, fontweight='bold')
    ax.set_title('Critical Difference Diagram (Statistical Ranking)', fontsize=14, fontweight='bold')
    ax.set_xlim([0, 3])
    ax.grid(True, alpha=0.3, axis='x')

    plt.tight_layout()
    plt.savefig(output_path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"Saved: {output_path}")


print("Plotting functions defined.")

In [ ]:
# Generate all required plots
print("Generating plots...\n")

# Graph 1: Accuracy vs Distance
plot_accuracy_vs_distance(
    accuracy_results,
    statistical_tests,
    PLOTS_DIR / "accuracy_vs_distance.png"
)

# Graph 2: Pareto Frontier
plot_pareto_frontier(
    accuracy_results,
    inference_benchmarks,
    PLOTS_DIR / "pareto_frontier.png"
)

# Graph 3: Convergence Curves
plot_convergence_curves(
    training_convergence,
    PLOTS_DIR / "convergence_curves.png"
)

# Graph 4: Throughput vs Batch Size
plot_throughput_vs_batchsize(
    inference_benchmarks,
    PLOTS_DIR / "throughput_vs_batchsize.png"
)

# Graph 5: Critical Difference
plot_critical_difference(
    accuracy_results,
    statistical_tests,
    PLOTS_DIR / "critical_difference.png"
)

print("\n✓ All plots generated")

## 13. Create Summary Table

In [ ]:
# Create comprehensive summary table
summary_data = []

for d in sorted(DISTANCES):
    if d not in accuracy_results or d not in inference_benchmarks:
        continue

    # DeepSets row
    if 'deepsets' in accuracy_results[d] and 'deepsets' in inference_benchmarks[d]:
        deepsets_acc = accuracy_results[d]['deepsets']
        deepsets_inf = inference_benchmarks[d]['deepsets']
        batch_size_64 = deepsets_inf.get(64, {})

        p_value = statistical_tests.get(d, {}).get('wilcoxon', {}).get('p_value', np.nan)
        effect_size = statistical_tests.get(d, {}).get('cohens_d', np.nan)

        summary_data.append({
            'Distance': d,
            'Model': 'DeepSets',
            'Params': deepsets_info.get(d, {}).get('num_parameters', 0),
            'Accuracy (mean±std)': f"{deepsets_acc['accuracy_mean']:.4f}±{deepsets_acc['accuracy_std']:.4f}",
            'F1 (mean±std)': f"{deepsets_acc['f1_mean']:.4f}±{deepsets_acc['f1_std']:.4f}",
            'Latency (μs, mean±std)': f"{batch_size_64.get('latency_per_sample_us', 0):.2f}±{batch_size_64.get('std_time_seconds', 0)*1e6/len(shared_test_data[d]['labels']):.2f}" if batch_size_64 else "N/A",
            'Throughput (samples/s)': f"{batch_size_64.get('throughput_samples_per_sec', 0):.0f}" if batch_size_64 else "N/A",
            'p-value vs DeepSets': '—',
            'Effect Size': '—'
        })

    # GraphSAGE row
    if 'graphsage' in accuracy_results[d] and 'graphsage' in inference_benchmarks[d]:
        graphsage_acc = accuracy_results[d]['graphsage']
        graphsage_inf = inference_benchmarks[d]['graphsage']
        batch_size_64 = graphsage_inf.get(64, {})

        p_value = statistical_tests.get(d, {}).get('wilcoxon', {}).get('p_value', np.nan)
        effect_size = statistical_tests.get(d, {}).get('cohens_d', np.nan)

        summary_data.append({
            'Distance': d,
            'Model': 'GraphSAGE',
            'Params': graphsage_info.get(d, {}).get('num_parameters', 0),
            'Accuracy (mean±std)': f"{graphsage_acc['accuracy_mean']:.4f}±{graphsage_acc['accuracy_std']:.4f}",
            'F1 (mean±std)': f"{graphsage_acc['f1_mean']:.4f}±{graphsage_acc['f1_std']:.4f}",
            'Latency (μs, mean±std)': f"{batch_size_64.get('latency_per_sample_us', 0):.2f}±{batch_size_64.get('std_time_seconds', 0)*1e6/len(shared_test_data[d]['labels']):.2f}" if batch_size_64 else "N/A",
            'Throughput (samples/s)': f"{batch_size_64.get('throughput_samples_per_sec', 0):.0f}" if batch_size_64 else "N/A",
            'p-value vs DeepSets': f"{p_value:.4f}" if not np.isnan(p_value) else "N/A",
            'Effect Size': f"{effect_size:.3f}" if not np.isnan(effect_size) else "N/A"
        })

summary_df = pd.DataFrame(summary_data)
summary_df.to_csv(OUTPUT_RESULTS_DIR / "summary_table.csv", index=False)

print("Summary Table:")
print(summary_df.to_string(index=False))
print(f"\n✓ Summary table saved to {OUTPUT_RESULTS_DIR / 'summary_table.csv'}")

## 14. Save Results and Generate Final Report

In [ ]:
# Prepare all results for saving
comparison_results = {
    'metadata': {
        'timestamp': datetime.now().isoformat(),
        'distances': DISTANCES,
        'test_samples_per_distance': TEST_SAMPLES_PER_DISTANCE,
        'num_benchmark_runs': NUM_BENCHMARK_RUNS,
        'num_accuracy_runs': NUM_ACCURACY_RUNS,
        'significance_level': ALPHA,
        'seed': SEED
    },
    'parameter_comparison': param_comparison,
    'accuracy_results': {
        str(d): {
            'deepsets': {
                'accuracy_mean': float(accuracy_results[d]['deepsets']['accuracy_mean']),
                'accuracy_std': float(accuracy_results[d]['deepsets']['accuracy_std']),
                'accuracy_ci_95': float(accuracy_results[d]['deepsets']['accuracy_ci_95']),
                'f1_mean': float(accuracy_results[d]['deepsets']['f1_mean']),
                'f1_std': float(accuracy_results[d]['deepsets']['f1_std']),
                'precision_mean': float(accuracy_results[d]['deepsets']['precision_mean']),
                'recall_mean': float(accuracy_results[d]['deepsets']['recall_mean'])
            },
            'graphsage': {
                'accuracy_mean': float(accuracy_results[d]['graphsage']['accuracy_mean']),
                'accuracy_std': float(accuracy_results[d]['graphsage']['accuracy_std']),
                'accuracy_ci_95': float(accuracy_results[d]['graphsage']['accuracy_ci_95']),
                'f1_mean': float(accuracy_results[d]['graphsage']['f1_mean']),
                'f1_std': float(accuracy_results[d]['graphsage']['f1_std']),
                'precision_mean': float(accuracy_results[d]['graphsage']['precision_mean']),
                'recall_mean': float(accuracy_results[d]['graphsage']['recall_mean'])
            }
        } for d in accuracy_results.keys()
    },
    'model_info': {
        'deepsets': {str(d): deepsets_info[d] for d in deepsets_info.keys()},
        'graphsage': {str(d): graphsage_info[d] for d in graphsage_info.keys()}
    }
}

# Save comparison results
with open(OUTPUT_RESULTS_DIR / "comparison_results.json", 'w') as f:
    json.dump(comparison_results, f, indent=2)

# Save inference benchmarks (simplified for JSON)
inference_benchmarks_json = {}
for d in inference_benchmarks.keys():
    inference_benchmarks_json[str(d)] = {}
    for model_type in ['deepsets', 'graphsage']:
        if model_type in inference_benchmarks[d]:
            inference_benchmarks_json[str(d)][model_type] = {
                str(bs): {
                    'throughput_samples_per_sec': float(inference_benchmarks[d][model_type][bs]['throughput_samples_per_sec']),
                    'latency_per_sample_us': float(inference_benchmarks[d][model_type][bs]['latency_per_sample_us']),
                    'mean_time_seconds': float(inference_benchmarks[d][model_type][bs]['mean_time_seconds']),
                    'std_time_seconds': float(inference_benchmarks[d][model_type][bs]['std_time_seconds'])
                } for bs in inference_benchmarks[d][model_type].keys()
            }

with open(OUTPUT_RESULTS_DIR / "inference_benchmarks.json", 'w') as f:
    json.dump(inference_benchmarks_json, f, indent=2)

# Save statistical tests
# Convert numpy types to native Python types for JSON
def convert_to_json_serializable(obj):
    if isinstance(obj, dict):
        return {k: convert_to_json_serializable(v) for k, v in obj.items()}
    elif isinstance(obj, list):
        return [convert_to_json_serializable(item) for item in obj]
    elif isinstance(obj, (np.integer, np.floating)):
        return float(obj)
    elif isinstance(obj, np.ndarray):
        return obj.tolist()
    elif isinstance(obj, (np.bool_, bool)):
        return bool(obj)
    else:
        return obj

statistical_tests_json = convert_to_json_serializable(statistical_tests)
with open(OUTPUT_RESULTS_DIR / "statistical_tests.json", 'w') as f:
    json.dump(statistical_tests_json, f, indent=2)

print("✓ All results saved to JSON files")

In [ ]:
# Generate final summary report
report_lines = []
report_lines.append("=" * 80)
report_lines.append("DEEPSETS vs GRAPH NEURAL NETWORK COMPARISON REPORT")
report_lines.append("=" * 80)
report_lines.append(f"Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
report_lines.append("")

# Overall statistical test results
if 'overall' in statistical_tests:
    overall = statistical_tests['overall']
    wilcoxon_result = overall['wilcoxon']
    cohens_d_val = overall['cohens_d']

    report_lines.append("OVERALL STATISTICAL ANALYSIS")
    report_lines.append("-" * 80)
    report_lines.append(f"Wilcoxon Signed-Rank Test:")
    report_lines.append(f"  Statistic: {wilcoxon_result['statistic']:.4f}")
    report_lines.append(f"  p-value: {wilcoxon_result['p_value']:.6f}")
    report_lines.append(f"  Significant (α={ALPHA}): {'YES' if wilcoxon_result['significant'] else 'NO'}")
    report_lines.append(f"  Effect Size (Cohen's d): {cohens_d_val:.4f} ({overall['effect_size_interpretation']})")
    report_lines.append("")

# Per-distance summary
report_lines.append("PER-DISTANCE SUMMARY")
report_lines.append("-" * 80)
for d in sorted(accuracy_results.keys()):
    if 'deepsets' not in accuracy_results[d] or 'graphsage' not in accuracy_results[d]:
        continue

    deepsets_acc = accuracy_results[d]['deepsets']['accuracy_mean']
    graphsage_acc = accuracy_results[d]['graphsage']['accuracy_mean']

    if d in statistical_tests:
        p_val = statistical_tests[d]['wilcoxon']['p_value']
        sig = statistical_tests[d]['wilcoxon']['significant']
        effect = statistical_tests[d]['cohens_d']
    else:
        p_val = np.nan
        sig = False
        effect = np.nan

    report_lines.append(f"Distance d={d}:")
    report_lines.append(f"  DeepSets: {deepsets_acc:.4f} ± {accuracy_results[d]['deepsets']['accuracy_std']:.4f}")
    report_lines.append(f"  GraphSAGE: {graphsage_acc:.4f} ± {accuracy_results[d]['graphsage']['accuracy_std']:.4f}")
    report_lines.append(f"  Difference: {graphsage_acc - deepsets_acc:+.4f}")
    report_lines.append(f"  p-value: {p_val:.6f} {'(significant)' if sig else '(not significant)'}")
    report_lines.append(f"  Effect size: {effect:.4f}")
    report_lines.append("")

# Final verdict
report_lines.append("FINAL VERDICT")
report_lines.append("-" * 80)

if 'overall' in statistical_tests:
    overall_wilcoxon = statistical_tests['overall']['wilcoxon']
    overall_cohens_d = statistical_tests['overall']['cohens_d']

    # Calculate average accuracies
    deepsets_avg_acc = np.mean([accuracy_results[d]['deepsets']['accuracy_mean'] for d in accuracy_results.keys() if 'deepsets' in accuracy_results[d]])
    graphsage_avg_acc = np.mean([accuracy_results[d]['graphsage']['accuracy_mean'] for d in accuracy_results.keys() if 'graphsage' in accuracy_results[d]])

    # Calculate average throughput (batch size 64)
    deepsets_throughputs = []
    graphsage_throughputs = []
    for d in inference_benchmarks.keys():
        if 'deepsets' in inference_benchmarks[d] and 64 in inference_benchmarks[d]['deepsets']:
            deepsets_throughputs.append(inference_benchmarks[d]['deepsets'][64]['throughput_samples_per_sec'])
        if 'graphsage' in inference_benchmarks[d] and 64 in inference_benchmarks[d]['graphsage']:
            graphsage_throughputs.append(inference_benchmarks[d]['graphsage'][64]['throughput_samples_per_sec'])

    deepsets_avg_throughput = np.mean(deepsets_throughputs) if deepsets_throughputs else 0
    graphsage_avg_throughput = np.mean(graphsage_throughputs) if graphsage_throughputs else 0

    verdict = f"""GraphSAGE achieved {graphsage_avg_acc:.4f} accuracy (mean ± std) vs DeepSets's {deepsets_avg_acc:.4f} accuracy (mean ± std) across distances d=3 to d=13. The Wilcoxon signed-rank test showed this difference {'WAS' if overall_wilcoxon['significant'] else 'WAS NOT'} statistically significant (p = {overall_wilcoxon['p_value']:.6f}, effect size = {overall_cohens_d:.4f}). However, DeepSets achieved {deepsets_avg_throughput/graphsage_avg_throughput:.2f}x faster inference throughput ({deepsets_avg_throughput:.0f} samples/s vs {graphsage_avg_throughput:.0f} samples/s). For real-time QEC applications requiring low latency, {'DeepSets' if deepsets_avg_throughput > graphsage_avg_throughput else 'GraphSAGE'} is recommended based on the Pareto frontier analysis."""

    report_lines.append(verdict)

report_lines.append("")
report_lines.append("=" * 80)

# Save report
report_text = "\n".join(report_lines)
with open(OUTPUT_RESULTS_DIR / "summary_report.txt", 'w', encoding='utf-8') as f:
    f.write(report_text)

print("\n" + report_text)
print(f"\n✓ Summary report saved to {OUTPUT_RESULTS_DIR / 'summary_report.txt'}")

## 15. Final Publication Plots

This section generates publication-ready plots and exports all data to CSV files.

**Toggle behavior:**
- `USE_SAVED_DATA = False`: Runs full benchmarks (including MWPM), generates plots, saves CSVs
- `USE_SAVED_DATA = True`: Loads pre-saved CSVs from `/plots/final/`, regenerates plots only (fast)

**Outputs (saved to `/plots/final/`):**
- 4 PNG plots: accuracy vs distance, Wilcoxon dot plot, inference scaling, parameter scaling
- 4 CSV files: final_accuracies.csv, wilcoxon_results.csv, inference_speeds.csv, parameter_counts.csv

In [ ]:
# MWPM Inference Speed Benchmarking
# Only runs if USE_SAVED_DATA is False

import pymatching

def benchmark_mwpm_latency(d: int, num_samples: int = 10000, num_runs: int = 5) -> Dict:
    """
    Benchmark MWPM decoder inference latency for a given code distance.

    Args:
        d: Code distance
        num_samples: Number of samples to decode
        num_runs: Number of timed runs for statistical robustness

    Returns:
        Dict with latency and throughput statistics
    """
    # Generate circuit and detection events
    circuit = stim.Circuit.generated(
        "surface_code:rotated_memory_z",
        rounds=d,
        distance=d,
        after_clifford_depolarization=0.005,
    )

    sampler = circuit.compile_detector_sampler()
    detection_events, _ = sampler.sample(num_samples, separate_observables=True)

    # Create MWPM decoder
    detector_error_model = circuit.detector_error_model(decompose_errors=True)
    matcher = pymatching.Matching.from_detector_error_model(detector_error_model)

    # Warmup runs
    for _ in range(3):
        _ = matcher.decode_batch(detection_events[:100])

    # Timed runs
    run_times = []
    for _ in range(num_runs):
        start = time.time()
        _ = matcher.decode_batch(detection_events)
        elapsed = time.time() - start
        run_times.append(elapsed)

    mean_time = np.mean(run_times)
    std_time = np.std(run_times)
    latency_per_sample_us = (mean_time / num_samples) * 1e6
    throughput = num_samples / mean_time

    return {
        'mean_time_seconds': mean_time,
        'std_time_seconds': std_time,
        'latency_per_sample_us': latency_per_sample_us,
        'throughput_samples_per_sec': throughput,
        'num_samples': num_samples,
        'num_runs': num_runs
    }

# Benchmark MWPM if not using saved data
mwpm_benchmarks = {}

if not USE_SAVED_DATA:
    print("Benchmarking MWPM decoder inference speed...")
    for d in DISTANCES:
        print(f"  Benchmarking MWPM d={d}...")
        mwpm_benchmarks[d] = benchmark_mwpm_latency(d, BENCHMARK_SAMPLES, NUM_BENCHMARK_RUNS)
        print(f"    Latency: {mwpm_benchmarks[d]['latency_per_sample_us']:.2f} us/sample")
    print("MWPM benchmarking complete")
else:
    print("Skipping MWPM benchmarking (USE_SAVED_DATA=True)")

In [ ]:
# Data Loading / Preparation
# Either load from saved CSVs or prepare from in-memory data

if USE_SAVED_DATA:
    # Load all CSVs from /final folder
    print("Loading data from saved CSVs in /plots/final/...")

    final_accuracies_df = pd.read_csv(FINAL_PLOTS_DIR / "final_accuracies.csv")
    wilcoxon_df = pd.read_csv(FINAL_PLOTS_DIR / "wilcoxon_results.csv")
    inference_speeds_df = pd.read_csv(FINAL_PLOTS_DIR / "inference_speeds.csv")
    parameter_counts_df = pd.read_csv(FINAL_PLOTS_DIR / "parameter_counts.csv")

    print(f"  Loaded final_accuracies.csv: {len(final_accuracies_df)} rows")
    print(f"  Loaded wilcoxon_results.csv: {len(wilcoxon_df)} rows")
    print(f"  Loaded inference_speeds.csv: {len(inference_speeds_df)} rows")
    print(f"  Loaded parameter_counts.csv: {len(parameter_counts_df)} rows")
    print("Data loaded from saved CSVs")

else:
    # Prepare DataFrames from in-memory data (computed earlier in notebook)
    print("Preparing data from in-memory results...")

    # 1. Final Accuracies DataFrame
    accuracies_data = []
    for d in DISTANCES:
        if d in accuracy_results:
            if 'deepsets' in accuracy_results[d]:
                r = accuracy_results[d]['deepsets']
                accuracies_data.append({
                    'distance': d,
                    'model': 'DeepSets',
                    'accuracy': r['accuracy_mean'],
                    'accuracy_std': r['accuracy_std'],
                    'f1': r['f1_mean'],
                    'precision': r['precision_mean'],
                    'recall': r['recall_mean']
                })
            if 'graphsage' in accuracy_results[d]:
                r = accuracy_results[d]['graphsage']
                accuracies_data.append({
                    'distance': d,
                    'model': 'GNN',
                    'accuracy': r['accuracy_mean'],
                    'accuracy_std': r['accuracy_std'],
                    'f1': r['f1_mean'],
                    'precision': r['precision_mean'],
                    'recall': r['recall_mean']
                })
    final_accuracies_df = pd.DataFrame(accuracies_data)

    # 2. Wilcoxon Results DataFrame
    wilcoxon_data = []
    if 'overall' in statistical_tests:
        overall = statistical_tests['overall']
        wilcoxon_data.append({
            'test_type': 'pooled',
            'statistic': overall['wilcoxon']['statistic'],
            'p_value': overall['wilcoxon']['p_value'],
            'significant': overall['wilcoxon']['significant'],
            'effect_size_r': overall['wilcoxon']['effect_size_r'],
            'cohens_d': overall['cohens_d'],
            'interpretation': overall['effect_size_interpretation'],
            'n_observations': len(deepsets_all_accs)
        })
    # Add per-distance results
    for d in DISTANCES:
        if d in statistical_tests:
            st = statistical_tests[d]
            wilcoxon_data.append({
                'test_type': f'distance_{d}',
                'statistic': st['wilcoxon']['statistic'],
                'p_value': st['wilcoxon']['p_value'],
                'significant': st['wilcoxon']['significant'],
                'effect_size_r': st['wilcoxon']['effect_size_r'],
                'cohens_d': st['cohens_d'],
                'interpretation': st['effect_size_interpretation'],
                'n_observations': NUM_ACCURACY_RUNS
            })
    wilcoxon_df = pd.DataFrame(wilcoxon_data)

    # 3. Inference Speeds DataFrame
    inference_data = []
    batch_size = 64  # Use batch size 64 as representative
    for d in DISTANCES:
        if d in inference_benchmarks:
            # NN
            if 'deepsets' in inference_benchmarks[d] and batch_size in inference_benchmarks[d]['deepsets']:
                r = inference_benchmarks[d]['deepsets'][batch_size]
                inference_data.append({
                    'distance': d,
                    'model': 'DeepSets',
                    'latency_us': r['latency_per_sample_us'],
                    'throughput_samples_per_sec': r['throughput_samples_per_sec'],
                    'batch_size': batch_size
                })
            # GNN
            if 'graphsage' in inference_benchmarks[d] and batch_size in inference_benchmarks[d]['graphsage']:
                r = inference_benchmarks[d]['graphsage'][batch_size]
                inference_data.append({
                    'distance': d,
                    'model': 'GNN',
                    'latency_us': r['latency_per_sample_us'],
                    'throughput_samples_per_sec': r['throughput_samples_per_sec'],
                    'batch_size': batch_size
                })
        # MWPM
        if d in mwpm_benchmarks:
            r = mwpm_benchmarks[d]
            inference_data.append({
                'distance': d,
                'model': 'MWPM',
                'latency_us': r['latency_per_sample_us'],
                'throughput_samples_per_sec': r['throughput_samples_per_sec'],
                'batch_size': 'N/A'
            })
    inference_speeds_df = pd.DataFrame(inference_data)

    # 4. Parameter Counts DataFrame
    parameter_counts_df = param_df.copy()
    parameter_counts_df = parameter_counts_df.rename(columns={
        'deepsets_params': 'deepsets_params',
        'graphsage_params': 'gdeepsets_params',
        'deepsets_size_mb': 'deepsets_size_mb',
        'graphsage_size_mb': 'gdeepsets_size_mb'
    })

    print(f"  Prepared final_accuracies_df: {len(final_accuracies_df)} rows")
    print(f"  Prepared wilcoxon_df: {len(wilcoxon_df)} rows")
    print(f"  Prepared inference_speeds_df: {len(inference_speeds_df)} rows")
    print(f"  Prepared parameter_counts_df: {len(parameter_counts_df)} rows")
    print("Data prepared from in-memory results")

In [ ]:
# Plot 1: Accuracy vs Distance

def plot_final_accuracy_vs_distance(df: pd.DataFrame, output_path: Path):
    """
    Plot accuracy vs code distance for NN and GNN models.

    Args:
        df: DataFrame with columns [distance, model, accuracy, accuracy_std]
        output_path: Path to save the plot
    """
    fig, ax = plt.subplots(figsize=(10, 6))

    # Get data for each model
    nn_data = df[df['model'] == 'DeepSets'].sort_values('distance')
    gnn_data = df[df['model'] == 'GNN'].sort_values('distance')

    # Plot NN
    ax.plot(nn_data['distance'], nn_data['accuracy'],
            'o-', label='DeepSets (DeepSets)', color='steelblue',
            linewidth=2.5, markersize=10, markeredgecolor='white', markeredgewidth=1.5)

    # Plot GNN
    ax.plot(gnn_data['distance'], gnn_data['accuracy'],
            's-', label='GNN (GraphSAGE)', color='coral',
            linewidth=2.5, markersize=10, markeredgecolor='white', markeredgewidth=1.5)

    # Styling
    ax.set_xlabel('Code Distance (d)', fontsize=14, fontweight='bold')
    ax.set_ylabel('Accuracy', fontsize=14, fontweight='bold')
    ax.set_title('Decoder Accuracy vs Code Distance', fontsize=16, fontweight='bold')
    ax.set_xticks(nn_data['distance'])
    ax.set_xticklabels([f'd={d}' for d in nn_data['distance']], fontsize=11)
    ax.tick_params(axis='y', labelsize=11)
    ax.legend(fontsize=12, loc='lower left')
    ax.grid(True, alpha=0.3)
    ax.set_ylim([0.5, 1.02])

    # Add accuracy values as annotations
    for _, row in nn_data.iterrows():
        ax.annotate(f'{row["accuracy"]:.3f}',
                   (row['distance'], row['accuracy']),
                   textcoords="offset points", xytext=(0, 10),
                   ha='center', fontsize=9, color='steelblue')
    for _, row in gnn_data.iterrows():
        ax.annotate(f'{row["accuracy"]:.3f}',
                   (row['distance'], row['accuracy']),
                   textcoords="offset points", xytext=(0, -15),
                   ha='center', fontsize=9, color='coral')

    plt.tight_layout()
    plt.savefig(output_path, dpi=300, bbox_inches='tight', facecolor='white')
    plt.close()
    print(f"Saved: {output_path}")

print("Plot function defined: plot_final_accuracy_vs_distance")

In [ ]:
# Plot 2: Wilcoxon Paired Differences Dot Plot

def plot_wilcoxon_paired_differences(accuracy_df: pd.DataFrame, wilcoxon_df: pd.DataFrame, output_path: Path):
    """
    Plot paired accuracy differences (GNN - DeepSets) with Wilcoxon test annotation.

    Shows the difference at each distance to visualize where GNN outperforms NN.

    Args:
        accuracy_df: DataFrame with columns [distance, model, accuracy]
        wilcoxon_df: DataFrame with Wilcoxon test results
        output_path: Path to save the plot
    """
    fig, ax = plt.subplots(figsize=(10, 6))

    # Calculate differences (GNN - DeepSets) for each distance
    differences = []
    distances_list = []

    for d in sorted(accuracy_df['distance'].unique()):
        nn_acc = accuracy_df[(accuracy_df['distance'] == d) & (accuracy_df['model'] == 'DeepSets')]['accuracy'].values
        gnn_acc = accuracy_df[(accuracy_df['distance'] == d) & (accuracy_df['model'] == 'GNN')]['accuracy'].values
        if len(nn_acc) > 0 and len(gnn_acc) > 0:
            diff = gnn_acc[0] - nn_acc[0]
            differences.append(diff)
            distances_list.append(d)

    # Create color palette based on distance
    colors = plt.cm.viridis(np.linspace(0.2, 0.8, len(distances_list)))

    # Plot differences as dots with varying sizes based on magnitude
    sizes = [abs(d) * 3000 + 100 for d in differences]
    scatter = ax.scatter(distances_list, differences, c=colors, s=sizes,
                        edgecolors='black', linewidth=1.5, alpha=0.8, zorder=3)

    # Connect with line
    ax.plot(distances_list, differences, '--', color='gray', alpha=0.5, linewidth=1.5, zorder=2)

    # Add reference line at 0
    ax.axhline(y=0, color='red', linestyle='-', linewidth=2, alpha=0.7, label='No difference')

    # Add annotation for each point
    for i, (d, diff) in enumerate(zip(distances_list, differences)):
        offset = 15 if diff >= 0 else -20
        ax.annotate(f'{diff:+.4f}', (d, diff),
                   textcoords="offset points", xytext=(0, offset),
                   ha='center', fontsize=10, fontweight='bold')

    # Get pooled Wilcoxon results
    pooled = wilcoxon_df[wilcoxon_df['test_type'] == 'pooled']
    if len(pooled) > 0:
        p_value = pooled['p_value'].values[0]
        cohens_d = pooled['cohens_d'].values[0]
        effect_r = pooled['effect_size_r'].values[0]
        sig_text = "SIGNIFICANT" if pooled['significant'].values[0] else "Not significant"

        # Add text box with statistics
        textstr = f'Pooled Wilcoxon Signed-Rank Test\n'
        textstr += f'p = {p_value:.6f} ({sig_text})\n'
        textstr += f"Cohen's d = {cohens_d:.3f}\n"
        textstr += f'Effect size r = {effect_r:.3f}'

        props = dict(boxstyle='round', facecolor='wheat', alpha=0.8)
        ax.text(0.02, 0.98, textstr, transform=ax.transAxes, fontsize=11,
               verticalalignment='top', bbox=props)

    # Styling
    ax.set_xlabel('Code Distance (d)', fontsize=14, fontweight='bold')
    ax.set_ylabel('Accuracy Difference (GNN - DeepSets)', fontsize=14, fontweight='bold')
    ax.set_title('Paired Accuracy Differences: GNN vs NN\n(Positive = GDeepSets better, Negative = DeepSets better)',
                fontsize=14, fontweight='bold')
    ax.set_xticks(distances_list)
    ax.set_xticklabels([f'd={d}' for d in distances_list], fontsize=11)
    ax.tick_params(axis='y', labelsize=11)
    ax.grid(True, alpha=0.3)
    ax.legend(loc='lower right', fontsize=11)

    # Shade regions
    ax.fill_between(ax.get_xlim(), 0, ax.get_ylim()[1], alpha=0.1, color='green', label='GDeepSets better')
    ax.fill_between(ax.get_xlim(), ax.get_ylim()[0], 0, alpha=0.1, color='blue', label='DeepSets better')

    plt.tight_layout()
    plt.savefig(output_path, dpi=300, bbox_inches='tight', facecolor='white')
    plt.close()
    print(f"Saved: {output_path}")

print("Plot function defined: plot_wilcoxon_paired_differences")

In [ ]:
# Plot 3: Inference Speed Scaling (NN, GNN, MWPM)

def plot_inference_scaling(df: pd.DataFrame, output_path: Path):
    """
    Plot inference latency vs code distance for NN, GNN, and MWPM decoders.

    Args:
        df: DataFrame with columns [distance, model, latency_us, throughput_samples_per_sec]
        output_path: Path to save the plot
    """
    fig, ax = plt.subplots(figsize=(10, 6))

    # Define styles for each model
    model_styles = {
        'DeepSets': {'color': 'steelblue', 'marker': 'o', 'label': 'NN (DeepSets)'},
        'GNN': {'color': 'coral', 'marker': 's', 'label': 'GNN (GraphSAGE)'},
        'MWPM': {'color': 'forestgreen', 'marker': '^', 'label': 'MWPM (PyMatching)'}
    }

    for model, style in model_styles.items():
        model_data = df[df['model'] == model].sort_values('distance')
        if len(model_data) > 0:
            ax.plot(model_data['distance'], model_data['latency_us'],
                   f"{style['marker']}-", label=style['label'], color=style['color'],
                   linewidth=2.5, markersize=10, markeredgecolor='white', markeredgewidth=1.5)

    # Styling
    ax.set_xlabel('Code Distance (d)', fontsize=14, fontweight='bold')
    ax.set_ylabel('Inference Latency (microseconds per sample)', fontsize=14, fontweight='bold')
    ax.set_title('Inference Latency Scaling: NN vs GNN vs MWPM', fontsize=16, fontweight='bold')

    distances = sorted(df['distance'].unique())
    ax.set_xticks(distances)
    ax.set_xticklabels([f'd={d}' for d in distances], fontsize=11)
    ax.tick_params(axis='y', labelsize=11)
    ax.legend(fontsize=12, loc='upper left')
    ax.grid(True, alpha=0.3)
    ax.set_yscale('log')

    # Add annotation for fastest decoder at d=13
    d13_data = df[df['distance'] == 13]
    if len(d13_data) > 0:
        fastest = d13_data.loc[d13_data['latency_us'].idxmin()]
        slowest = d13_data.loc[d13_data['latency_us'].idxmax()]
        speedup = slowest['latency_us'] / fastest['latency_us']

        textstr = f"At d=13:\n{fastest['model']}: {fastest['latency_us']:.1f} us\n"
        textstr += f"{slowest['model']}: {slowest['latency_us']:.1f} us\n"
        textstr += f"Speedup: {speedup:.1f}x"

        props = dict(boxstyle='round', facecolor='lightyellow', alpha=0.8)
        ax.text(0.98, 0.98, textstr, transform=ax.transAxes, fontsize=10,
               verticalalignment='top', horizontalalignment='right', bbox=props)

    plt.tight_layout()
    plt.savefig(output_path, dpi=300, bbox_inches='tight', facecolor='white')
    plt.close()
    print(f"Saved: {output_path}")

print("Plot function defined: plot_inference_scaling")

In [ ]:
# Plot 4: Parameter Scaling

def plot_parameter_scaling(df: pd.DataFrame, output_path: Path):
    """
    Plot number of parameters vs code distance for NN and GNN.

    Highlights that GNN has constant parameters while NN scales with distance.

    Args:
        df: DataFrame with columns [distance, deepsets_params, gdeepsets_params]
        output_path: Path to save the plot
    """
    fig, ax = plt.subplots(figsize=(10, 6))

    df_sorted = df.sort_values('distance')

    # Plot NN parameters
    ax.plot(df_sorted['distance'], df_sorted['deepsets_params'] / 1e6,
            'o-', label='DeepSets (DeepSets)', color='steelblue',
            linewidth=2.5, markersize=10, markeredgecolor='white', markeredgewidth=1.5)

    # Plot GNN parameters
    ax.plot(df_sorted['distance'], df_sorted['gdeepsets_params'] / 1e6,
            's-', label='GNN (GraphSAGE)', color='coral',
            linewidth=2.5, markersize=10, markeredgecolor='white', markeredgewidth=1.5)

    # Add annotations
    for _, row in df_sorted.iterrows():
        ax.annotate(f'{row["deepsets_params"]/1e6:.2f}M',
                   (row['distance'], row['deepsets_params']/1e6),
                   textcoords="offset points", xytext=(0, 10),
                   ha='center', fontsize=9, color='steelblue')
        ax.annotate(f'{row["gdeepsets_params"]/1e6:.2f}M',
                   (row['distance'], row['gdeepsets_params']/1e6),
                   textcoords="offset points", xytext=(0, -15),
                   ha='center', fontsize=9, color='coral')

    # Styling
    ax.set_xlabel('Code Distance (d)', fontsize=14, fontweight='bold')
    ax.set_ylabel('Number of Parameters (Millions)', fontsize=14, fontweight='bold')
    ax.set_title('Model Parameter Scaling: NN vs GNN', fontsize=16, fontweight='bold')

    ax.set_xticks(df_sorted['distance'])
    ax.set_xticklabels([f'd={d}' for d in df_sorted['distance']], fontsize=11)
    ax.tick_params(axis='y', labelsize=11)
    ax.legend(fontsize=12, loc='upper left')
    ax.grid(True, alpha=0.3)
    ax.set_yscale('log')

    # Add text box with key insight
    nn_avg = df_sorted['deepsets_params'].mean() / 1e6
    gnn_avg = df_sorted['gdeepsets_params'].mean() / 1e6
    ratio = nn_avg / gnn_avg

    textstr = f'Key Insight:\n'
    textstr += f'NN avg: {nn_avg:.2f}M params\n'
    textstr += f'GDeepSets: {gnn_avg:.2f}M params (constant)\n'
    textstr += f'DeepSets is {ratio:.0f}x larger on average\n'
    textstr += f'GDeepSets: Distance-agnostic architecture'

    props = dict(boxstyle='round', facecolor='lightcyan', alpha=0.8)
    ax.text(0.98, 0.02, textstr, transform=ax.transAxes, fontsize=10,
           verticalalignment='bottom', horizontalalignment='right', bbox=props)

    plt.tight_layout()
    plt.savefig(output_path, dpi=300, bbox_inches='tight', facecolor='white')
    plt.close()
    print(f"Saved: {output_path}")

print("Plot function defined: plot_parameter_scaling")

In [ ]:
# CSV Export Functions

def export_all_csvs(
    accuracies_df: pd.DataFrame,
    wilcoxon_df: pd.DataFrame,
    inference_df: pd.DataFrame,
    params_df: pd.DataFrame,
    output_dir: Path
):
    """
    Export all data to CSV files in the output directory.

    Args:
        accuracies_df: Accuracy results DataFrame
        wilcoxon_df: Wilcoxon test results DataFrame
        inference_df: Inference speed DataFrame
        params_df: Parameter counts DataFrame
        output_dir: Directory to save CSV files
    """
    # Save all CSVs
    accuracies_df.to_csv(output_dir / "final_accuracies.csv", index=False)
    print(f"Saved: {output_dir / 'final_accuracies.csv'}")

    wilcoxon_df.to_csv(output_dir / "wilcoxon_results.csv", index=False)
    print(f"Saved: {output_dir / 'wilcoxon_results.csv'}")

    inference_df.to_csv(output_dir / "inference_speeds.csv", index=False)
    print(f"Saved: {output_dir / 'inference_speeds.csv'}")

    params_df.to_csv(output_dir / "parameter_counts.csv", index=False)
    print(f"Saved: {output_dir / 'parameter_counts.csv'}")

print("CSV export function defined: export_all_csvs")

In [ ]:
# Generate All Final Outputs
# This cell generates all 4 plots and exports CSVs (if USE_SAVED_DATA=False)

print("=" * 60)
print("GENERATING FINAL PUBLICATION PLOTS")
print("=" * 60)
print(f"Output directory: {FINAL_PLOTS_DIR}")
print(f"USE_SAVED_DATA: {USE_SAVED_DATA}")
print()

# Export CSVs (only if we computed fresh data)
if not USE_SAVED_DATA:
    print("Exporting data to CSV files...")
    export_all_csvs(
        final_accuracies_df,
        wilcoxon_df,
        inference_speeds_df,
        parameter_counts_df,
        FINAL_PLOTS_DIR
    )
    print()

# Generate all plots
print("Generating plots...")

# Plot 1: Accuracy vs Distance
plot_final_accuracy_vs_distance(
    final_accuracies_df,
    FINAL_PLOTS_DIR / "accuracy_vs_distance.png"
)

# Plot 2: Wilcoxon Paired Differences
plot_wilcoxon_paired_differences(
    final_accuracies_df,
    wilcoxon_df,
    FINAL_PLOTS_DIR / "wilcoxon_paired_differences.png"
)

# Plot 3: Inference Speed Scaling
plot_inference_scaling(
    inference_speeds_df,
    FINAL_PLOTS_DIR / "inference_scaling.png"
)

# Plot 4: Parameter Scaling
plot_parameter_scaling(
    parameter_counts_df,
    FINAL_PLOTS_DIR / "parameter_scaling.png"
)

print()
print("=" * 60)
print("FINAL OUTPUTS COMPLETE")
print("=" * 60)
print(f"\nPlots saved to: {FINAL_PLOTS_DIR}")
print("  - accuracy_vs_distance.png")
print("  - wilcoxon_paired_differences.png")
print("  - inference_scaling.png")
print("  - parameter_scaling.png")

if not USE_SAVED_DATA:
    print("\nCSVs saved to: {FINAL_PLOTS_DIR}")
    print("  - final_accuracies.csv")
    print("  - wilcoxon_results.csv")
    print("  - inference_speeds.csv")
    print("  - parameter_counts.csv")
else:
    print("\n(CSVs not exported - using saved data)")

In [ ]:
# Inference Speed Scaling Pattern Analysis
# Characterize the scaling behavior: Is it linear, sub-linear (logarithmic), or super-linear (polynomial)?

from scipy.optimize import curve_fit
from sklearn.metrics import r2_score

print("=" * 70)
print("INFERENCE SPEED SCALING PATTERN ANALYSIS")
print("=" * 70)
print("\nQuestion: How does inference latency scale with code distance?")
print("  - Linear scaling: latency grows proportionally with distance")
print("  - Sub-linear (logarithmic): latency grows slower than distance (more efficient)")
print("  - Super-linear (polynomial): latency grows faster than distance (less efficient)")
print()

# Define scaling models
def linear(d, a, b):
    return a * d + b

def logarithmic(d, a, b):
    return a * np.log(d) + b

def quadratic(d, a, b, c):
    return a * d**2 + b * d + c

def power_law(d, a, b):
    return a * np.power(d.astype(float), b)

# Analyze each decoder
scaling_results = {}

for model_type in ['DeepSets', 'GNN', 'MWPM']:
    model_data = inference_speeds_df[inference_speeds_df['model'] == model_type].sort_values('distance')

    if len(model_data) < 3:
        continue

    d = model_data['distance'].values.astype(float)
    lat = model_data['latency_us'].values.astype(float)

    # Fit each model
    fits = {}

    # Linear fit
    try:
        popt, _ = curve_fit(linear, d, lat)
        pred = linear(d, *popt)
        fits['linear'] = {'params': popt, 'r2': r2_score(lat, pred), 'slope': popt[0]}
    except:
        fits['linear'] = {'params': None, 'r2': -np.inf}

    # Logarithmic fit
    try:
        popt, _ = curve_fit(logarithmic, d, lat, p0=[10, 0])
        pred = logarithmic(d, *popt)
        fits['logarithmic'] = {'params': popt, 'r2': r2_score(lat, pred)}
    except:
        fits['logarithmic'] = {'params': None, 'r2': -np.inf}

    # Quadratic fit
    try:
        popt, _ = curve_fit(quadratic, d, lat)
        pred = quadratic(d, *popt)
        fits['quadratic'] = {'params': popt, 'r2': r2_score(lat, pred)}
    except:
        fits['quadratic'] = {'params': None, 'r2': -np.inf}

    # Power law fit (f(d) = a * d^b) - exponent b tells us the scaling
    try:
        popt, _ = curve_fit(power_law, d, lat, p0=[1, 1], maxfev=5000)
        pred = power_law(d, *popt)
        fits['power_law'] = {'params': popt, 'r2': r2_score(lat, pred), 'exponent': popt[1]}
    except:
        fits['power_law'] = {'params': None, 'r2': -np.inf}

    # Compute growth factor (how much latency increases from d=3 to d=13)
    lat_d3 = model_data[model_data['distance'] == 3]['latency_us'].values
    lat_d13 = model_data[model_data['distance'] == 13]['latency_us'].values
    growth_factor = lat_d13[0] / lat_d3[0] if len(lat_d3) > 0 and len(lat_d13) > 0 else np.nan

    # Determine scaling pattern
    r2_lin = fits['linear']['r2']
    r2_log = fits['logarithmic']['r2']
    r2_quad = fits['quadratic']['r2']

    # Classification based on R² comparison and power law exponent
    if fits['power_law']['params'] is not None:
        exponent = fits['power_law']['exponent']
        if exponent < 0.5:
            pattern = "SUB-LINEAR (nearly constant)"
        elif exponent < 0.9:
            pattern = "SUB-LINEAR (logarithmic-like)"
        elif exponent < 1.1:
            pattern = "LINEAR"
        elif exponent < 2.0:
            pattern = "SUPER-LINEAR (polynomial)"
        else:
            pattern = "SUPER-LINEAR (quadratic+)"
    else:
        if r2_log > r2_lin and r2_log > r2_quad:
            pattern = "SUB-LINEAR (logarithmic)"
        elif r2_quad > r2_lin:
            pattern = "SUPER-LINEAR (polynomial)"
        else:
            pattern = "LINEAR"
        exponent = 1.0

    scaling_results[model_type] = {
        'fits': fits,
        'growth_factor': growth_factor,
        'pattern': pattern,
        'exponent': exponent,
        'latency_d3': lat_d3[0] if len(lat_d3) > 0 else np.nan,
        'latency_d13': lat_d13[0] if len(lat_d13) > 0 else np.nan
    }

# Print results
for model_type in ['DeepSets', 'GNN', 'MWPM']:
    if model_type not in scaling_results:
        continue

    res = scaling_results[model_type]
    fits = res['fits']

    print(f"\n{'='*50}")
    print(f"{model_type} DECODER")
    print(f"{'='*50}")
    print(f"\n  Scaling Pattern: {res['pattern']}")
    print(f"  Power Law Exponent: d^{res['exponent']:.2f}")
    print(f"\n  Growth from d=3 to d=13:")
    print(f"    Latency at d=3:  {res['latency_d3']:.2f} μs")
    print(f"    Latency at d=13: {res['latency_d13']:.2f} μs")
    print(f"    Growth Factor:   {res['growth_factor']:.2f}x")
    print(f"\n  Model Fit Comparison (R²):")
    print(f"    Linear:      R² = {fits['linear']['r2']:.4f}")
    print(f"    Logarithmic: R² = {fits['logarithmic']['r2']:.4f}")
    print(f"    Quadratic:   R² = {fits['quadratic']['r2']:.4f}")
    print(f"    Power Law:   R² = {fits['power_law']['r2']:.4f}")

# Summary comparison
print("\n" + "=" * 70)
print("SCALING PATTERN SUMMARY")
print("=" * 70)

summary_data = []
for model_type in ['DeepSets', 'GNN', 'MWPM']:
    if model_type in scaling_results:
        res = scaling_results[model_type]
        summary_data.append({
            'Decoder': model_type,
            'Scaling Pattern': res['pattern'],
            'Exponent (d^n)': f"{res['exponent']:.2f}",
            'Growth (d3→d13)': f"{res['growth_factor']:.2f}x",
            'Latency d=3 (μs)': f"{res['latency_d3']:.1f}",
            'Latency d=13 (μs)': f"{res['latency_d13']:.1f}"
        })

summary_df = pd.DataFrame(summary_data)
print(summary_df.to_string(index=False))

# Save to CSV
summary_df.to_csv(FINAL_PLOTS_DIR / "scaling_patterns.csv", index=False)
print(f"\nSaved: {FINAL_PLOTS_DIR / 'scaling_patterns.csv'}")

In [ ]:
# Plot Scaling Patterns with Power Law Fits

def plot_scaling_patterns(df: pd.DataFrame, scaling_results: Dict, output_path: Path):
    """
    Plot inference latency vs code distance with power law fits to show scaling patterns.
    """
    fig, ax = plt.subplots(figsize=(11, 7))

    model_styles = {
        'DeepSets': {'color': 'steelblue', 'marker': 'o'},
        'GNN': {'color': 'coral', 'marker': 's'},
        'MWPM': {'color': 'forestgreen', 'marker': '^'}
    }

    d_extended = np.linspace(2.5, 14, 100)

    for model_type in ['DeepSets', 'GNN', 'MWPM']:
        model_data = df[df['model'] == model_type].sort_values('distance')
        if len(model_data) == 0 or model_type not in scaling_results:
            continue

        style = model_styles[model_type]
        res = scaling_results[model_type]
        pattern = res['pattern']
        exponent = res['exponent']

        # Plot data points
        ax.scatter(model_data['distance'], model_data['latency_us'],
                  s=180, marker=style['marker'], color=style['color'],
                  edgecolors='black', linewidth=2, zorder=3,
                  label=f"{model_type}: {pattern}")

        # Plot power law fit
        fits = res['fits']
        if fits['power_law']['params'] is not None:
            params = fits['power_law']['params']
            y_fit = params[0] * np.power(d_extended, params[1])
            ax.plot(d_extended, y_fit, '--', color=style['color'],
                   linewidth=2.5, alpha=0.7, zorder=2)

    ax.set_xlabel('Code Distance (d)', fontsize=14, fontweight='bold')
    ax.set_ylabel('Inference Latency (μs per sample)', fontsize=14, fontweight='bold')
    ax.set_title('Decoder Latency Scaling Patterns', fontsize=16, fontweight='bold')

    ax.set_yscale('log')
    ax.grid(True, alpha=0.3, which='both')
    ax.legend(fontsize=11, loc='upper left')

    # Add scaling interpretation box
    textstr = "Scaling Analysis (Power Law: d^n)\n"
    textstr += "-" * 30 + "\n"
    for model_type in ['DeepSets', 'GNN', 'MWPM']:
        if model_type in scaling_results:
            res = scaling_results[model_type]
            textstr += f"{model_type}: d^{res['exponent']:.2f} ({res['growth_factor']:.1f}x growth)\n"

    textstr += "\nInterpretation:\n"
    textstr += "n < 1: Sub-linear (efficient)\n"
    textstr += "n = 1: Linear\n"
    textstr += "n > 1: Super-linear (inefficient)"

    props = dict(boxstyle='round', facecolor='lightyellow', alpha=0.95, edgecolor='gray')
    ax.text(0.98, 0.02, textstr.strip(), transform=ax.transAxes, fontsize=10,
           verticalalignment='bottom', horizontalalignment='right',
           bbox=props, family='monospace')

    plt.tight_layout()
    plt.savefig(output_path, dpi=300, bbox_inches='tight', facecolor='white')
    plt.close()
    print(f"Saved: {output_path}")

# Generate the scaling pattern plot
plot_scaling_patterns(
    inference_speeds_df,
    scaling_results,
    FINAL_PLOTS_DIR / "scaling_patterns.png"
)

print("\nScaling pattern analysis complete!")

In [ ]:
# Practical Interpretation of Scaling Patterns

print("=" * 70)
print("PRACTICAL IMPLICATIONS OF SCALING PATTERNS")
print("=" * 70)

# Analyze the scaling patterns
nn_exp = scaling_results['DeepSets']['exponent'] if 'DeepSets' in scaling_results else None
gnn_exp = scaling_results['GNN']['exponent'] if 'GNN' in scaling_results else None
mwpm_exp = scaling_results['MWPM']['exponent'] if 'MWPM' in scaling_results else None

print("\n1. SCALING BEHAVIOR SUMMARY")
print("-" * 50)

if nn_exp is not None:
    nn_growth = scaling_results['DeepSets']['growth_factor']
    if nn_exp < 0.5:
        nn_behavior = "nearly constant - excellent scalability"
    elif nn_exp < 0.9:
        nn_behavior = "sub-linear - good scalability"
    elif nn_exp < 1.1:
        nn_behavior = "approximately linear"
    else:
        nn_behavior = "super-linear - poor scalability"
    print(f"   DeepSets:   d^{nn_exp:.2f} - {nn_behavior}")
    print(f"         Latency grows {nn_growth:.1f}x from d=3 to d=13")

if gnn_exp is not None:
    gnn_growth = scaling_results['GNN']['growth_factor']
    if gnn_exp < 0.5:
        gnn_behavior = "nearly constant - excellent scalability"
    elif gnn_exp < 0.9:
        gnn_behavior = "sub-linear - good scalability"
    elif gnn_exp < 1.1:
        gnn_behavior = "approximately linear"
    else:
        gnn_behavior = "super-linear - poor scalability"
    print(f"   GDeepSets:  d^{gnn_exp:.2f} - {gnn_behavior}")
    print(f"         Latency grows {gnn_growth:.1f}x from d=3 to d=13")

if mwpm_exp is not None:
    mwpm_growth = scaling_results['MWPM']['growth_factor']
    if mwpm_exp < 0.5:
        mwpm_behavior = "nearly constant - excellent scalability"
    elif mwpm_exp < 0.9:
        mwpm_behavior = "sub-linear - good scalability"
    elif mwpm_exp < 1.1:
        mwpm_behavior = "approximately linear"
    else:
        mwpm_behavior = "super-linear - poor scalability"
    print(f"   MWPM: d^{mwpm_exp:.2f} - {mwpm_behavior}")
    print(f"         Latency grows {mwpm_growth:.1f}x from d=3 to d=13")

print("\n2. WHAT THIS MEANS FOR LARGE-SCALE QEC")
print("-" * 50)

# Compare scaling efficiency
decoders_by_efficiency = []
for name, exp in [('DeepSets', nn_exp), ('GNN', gnn_exp), ('MWPM', mwpm_exp)]:
    if exp is not None:
        decoders_by_efficiency.append((name, exp))

decoders_by_efficiency.sort(key=lambda x: x[1])  # Lower exponent = better scaling

if len(decoders_by_efficiency) >= 2:
    best = decoders_by_efficiency[0]
    worst = decoders_by_efficiency[-1]

    print(f"   Best scaling:  {best[0]} (d^{best[1]:.2f})")
    print(f"   Worst scaling: {worst[0]} (d^{worst[1]:.2f})")

    # Project to larger distances
    print("\n   Projected latency at d=21 (extrapolation):")
    for name in ['DeepSets', 'GNN', 'MWPM']:
        if name in scaling_results:
            res = scaling_results[name]
            params = res['fits']['power_law']['params']
            if params is not None:
                lat_d21 = params[0] * (21 ** params[1])
                lat_d3 = res['latency_d3']
                print(f"      {name}: ~{lat_d21:.0f} μs ({lat_d21/lat_d3:.1f}x growth from d=3)")

print("\n3. KEY TAKEAWAYS")
print("-" * 50)

# Generate takeaways based on actual results
takeaways = []

if mwpm_exp is not None and mwpm_exp < 1.0:
    takeaways.append(f"MWPM shows sub-linear scaling (d^{mwpm_exp:.2f}), meaning it becomes")
    takeaways.append("   relatively MORE efficient at larger code distances.")

if nn_exp is not None and nn_exp >= 1.0:
    takeaways.append(f"NN shows approximately linear scaling (d^{nn_exp:.2f}), meaning latency")
    takeaways.append("   grows proportionally with code distance.")

if gnn_exp is not None:
    if gnn_exp > 1.5:
        takeaways.append(f"GNN shows super-linear scaling (d^{gnn_exp:.2f}), meaning it becomes")
        takeaways.append("   less efficient at larger code distances despite constant parameters.")
    elif gnn_exp < 1.0:
        takeaways.append(f"GNN shows sub-linear scaling (d^{gnn_exp:.2f}), demonstrating")
        takeaways.append("   efficiency benefits from its graph-based architecture.")

# Compare absolute speeds vs scaling
if nn_exp is not None and gnn_exp is not None:
    nn_lat_d13 = scaling_results['DeepSets']['latency_d13']
    gnn_lat_d13 = scaling_results['GNN']['latency_d13']

    if nn_lat_d13 < gnn_lat_d13:
        takeaways.append(f"\nDespite any scaling advantages, NN remains {gnn_lat_d13/nn_lat_d13:.0f}x faster")
        takeaways.append(f"   than GNN in absolute terms at d=13.")

for t in takeaways:
    print(f"   {t}")

print("\n" + "=" * 70)